# Подготовка окружения и скачивание данных



In [1]:
# Установка необходимых библиотек
!pip install -q opendatasets lmdb fire natsort

import os
import opendatasets as od
# Скачиваем данные
od.download("https://www.kaggle.com/datasets/ziiiegen/price-ocr-data")

#Клонируем FORK репозиторий ClovaAI
!git clone https://github.com/ziiiegen/deep-text-recognition-benchmark.git
import sys
sys.path.append('/content/deep-text-recognition-benchmark')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.1/333.1 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 8.6 MB/s eta 0:00:00
Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: ziiiegen
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/ziiiegen/price-ocr-data


100%|██████████| 23.5M/23.5M [00:03<00:00, 7.57MB/s]



Cloning into 'deep-text-recognition-benchmark'...
remote: Enumerating objects: 542, done.
remote: Counting objects: 100% (268/268), done.
remote: Compressing objects: 100% (50/50), done.
remote: Total 542 (delta 237), reused 233 (delta 218), pack-reused 274 (from 1)
Receiving objects: 100% (542/542), 3.07 MiB | 7.34 MiB/s, done.
Resolving deltas: 100% (337/337), done.


# Подготовка LMDB датасетов (Train / Val)

In [ ]:
import os
import shutil
import lmdb
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split

TRAIN_CSV = "/content/price-ocr-data/OCR_DATA/train.csv"
TRAIN_IMG_DIR = "/content/price-ocr-data/OCR_DATA/train/train"

LMDB_TRAIN = "/content/data_lmdb/train"
LMDB_VAL = "/content/data_lmdb/val"

SEED = 40

# очищаем старые базы
for path in [LMDB_TRAIN, LMDB_VAL]:
    if os.path.exists(path):
        shutil.rmtree(path)
    os.makedirs(path, exist_ok=True)

df = pd.read_csv(TRAIN_CSV)
df = df.dropna(subset=["Filename", "Price"]).copy()

# приводим labels к строке из цифр
df["Price"] = df["Price"].astype(str).str.strip()
df["Price"] = df["Price"].str.replace(r"\.0$", "", regex=True)

df = df[df["Price"].str.fullmatch(r"\d+")].copy()

print("Total valid samples:", len(df))
print(df["Price"].head(10).tolist())

train_df, val_df = train_test_split(df, test_size=0.15, random_state=SEED)

def write_lmdb(df, img_dir, lmdb_path):
    env = lmdb.open(lmdb_path, map_size=1099511627776)
    cnt = 1

    with env.begin(write=True) as txn:
        for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Writing to {lmdb_path}"):
            img_path = os.path.join(img_dir, row["Filename"])

            if not os.path.exists(img_path):
                continue

            try:
                with open(img_path, "rb") as f:
                    imageBin = f.read()
            except:
                continue

            img = cv2.imdecode(np.frombuffer(imageBin, dtype=np.uint8), cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue

            label = str(row["Price"]).strip()

            if not label.isdigit():
                continue

            imageKey = f"image-{cnt:09d}".encode()
            labelKey = f"label-{cnt:09d}".encode()

            txn.put(imageKey, imageBin)
            txn.put(labelKey, label.encode("utf-8"))
            cnt += 1

        txn.put(b"num-samples", str(cnt - 1).encode())

    env.close()

write_lmdb(train_df, TRAIN_IMG_DIR, LMDB_TRAIN)
write_lmdb(val_df, TRAIN_IMG_DIR, LMDB_VAL)

Total valid samples: 18136
['66', '35', '57', '57', '25', '189', '46', '34', '1139', '34']


Writing to /content/data_lmdb/val: 100%|██████████| 2721/2721 [00:01<00:00, 2526.17it/s]


# Обучение модели

In [ ]:
!cd /content/deep-text-recognition-benchmark && python -u train.py \
--train_data /content/data_lmdb/train \
--valid_data /content/data_lmdb/val \
--manualSeed 42 \
--select_data "/" \
--batch_ratio 1.0 \
--num_fiducial 4 \
--Transformation TPS \
--FeatureExtraction ResNet \
--SequenceModeling BiLSTM \
--Prediction CTC \
--character "0123456789" \
--batch_max_length 6 \
--imgH 96 \
--imgW 110 \
--hidden_size 256 \
--PAD \
--batch_size 64 \
--workers 2 \
--adamW \
--lr 0.001 \
--num_iter 9000 \
--valInterval 200 \
--exp_name "ctc_size"

Filtering the images containing characters which are not in opt.character
Filtering the images whose label is longer than opt.batch_max_length
--------------------------------------------------------------------------------
dataset_root: /content/data_lmdb/train
opt.select_data: ['/']
opt.batch_ratio: ['1.0']
--------------------------------------------------------------------------------
dataset_root:    /content/data_lmdb/train	 dataset: /
sub-directory:	/.	 num samples: 15415
num total samples of /: 15415 x 1.0 (total_data_usage_ratio) = 15415
num samples of / per batch: 64 x 1.0 (batch_ratio) = 64
--------------------------------------------------------------------------------
Total_batch_size: 64 = 64
--------------------------------------------------------------------------------
dataset_root:    /content/data_lmdb/val	 dataset: /
sub-directory:	/.	 num samples: 2721
--------------------------------------------------------------------------------
model input parameters 96 110 4 1